In [1]:
import pandas as pd
import numpy as np
import io
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("--- Début du script d'entraînement ---")

csv_data = """microrna,microrna_group_simplified,parasite,organism,infection,cell type,time,is_upregulated
hsa-miR-22,miR-22,L.major,Human,in vitro,PBMC,3,1
mmu-miR-5115,miR-5115,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-596,miR-596,L.major,Human,in vitro,PBMC,2,1
mmu-miR-5119,miR-5119,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
mmu-miR-16,miR-16,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-let-7b,let-7b,L.major,Human,in vitro,PBMC,4,1
hsa-miR-29a,miR-29a,L.donovani,Human,in vitro,PBMC,9,0
mmu-miR-466p-3p,miR-466p,L.major,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-miR-30c-2,miR-30c,L.major,Human,in vitro,PBMC,2,1
hsa-miR-26a,miR-26a,L.major,Human,in vitro,PBMC,3,1
hsa-let-7a-1,let-7a,L.major,Human,in vitro,PBMC,4,1
mmu-miR-5102,miR-5102,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-495-3p,miR-495,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),24,1
mmu-miR-341,miR-341,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-1945,miR-1945,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-292-5p,miR-292,L.major,Mouse,in vitro,BMDM (BALB/c females),22,0
hsa-miR-30c-1,miR-30c,L.major,Human,in vitro,PBMC,3,1
hsa-miR-193a-5p,miR-193a,L.major,Human,in vitro,PBMC,8,0
mmu-miR-16,miR-16,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-107,miR-107,L.major,Human,in vitro,PBMC,26,1
hsa-miR-26b,miR-26b,L.major,Human,in vitro,PBMC,4,1
mmu-miR-144-3p,miR-144,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),22,0
hsa-miR-142-3p,miR-142,L.donovani,Human,in vitro,PBMC,8,1
mmu-miR-590-3p,miR-590,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),10,1
mmu-miR-3473b,miR-3473b,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
mmu-miR-295-3p,miR-295,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-142-3p,miR-142,L.major,Human,in vitro,PBMC,6,1
mmu-miR-155,miR-155,L.major,Mouse,in vitro,BMDM (BALB/c females),26,1
mmu-miR-1955-5p,miR-1955,L.major,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-miR-30a-5p,miR-30a,L.major,Human,in vitro,THP-1,10,1
mmu-miR-410-3p,miR-410,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),26,1
hsa-miR-29a,miR-29a,L.major,Human,in vitro,PBMC,13,1
hsa-miR-19b-2,miR-19b,L.major,Human,in vitro,PBMC,3,1
hsa-miR-107,miR-107,L.donovani,Human,in vitro,PBMC,10,0
mmu-miR-495-3p,miR-495,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),24,1
hsa-miR-210,miR-210,L.major,Human,in vitro,PBMC,24,1
mmu-miR-3068,miR-3068,L.major,Mouse,in vitro,BMDM (BALB/c females),22,0
mmu-miR-129-5p,miR-129,L.major,Mouse,in vitro,BMDM (BALB/c females),26,0
mmu-miR-144-3p,miR-144,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-331,miR-331,L.major,Human,in vitro,PBMC,1,1
hsa-miR-142-5p,miR-142,L.major,Human,in vitro,PBMC,8,1
hsa-miR-191-5p,miR-191,L.major,Human,in vitro,THP-1,7,1
mmu-miR-16,miR-16,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-1260,miR-1260,L.major,Human,in vitro,PBMC,6,1
hsa-miR-26b,miR-26b,L.major,Human,in vitro,PBMC,3,1
mmu-miR-221,miR-221,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-193a-5p,miR-193a,L.donovani,Human,in vitro,PBMC,9,1
hsa-miR-24,miR-24,L.major,Human,in vitro,PBMC,2,1
hsa-miR-22,miR-22,L.major,Human,in vitro,PBMC,8,0
mmu-miR-1955-5p,miR-1955,L.major,Mouse,in vitro,BMDM (BALB/c females),26,1
hsa-miR-210,miR-210,L.major,Human,in vitro,PBMC,12,1
mmu-let-7f,let-7f,L.major,Mouse,in vitro,BMDM (BALB/c females),26,1
mmu-miR-1892,miR-1892,L.major,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-miR-331-3p,miR-331,L.major,Human,in vitro,PBMC,8,0
hsa-let-7b-5p,let-7b,L.major,Human,in vitro,THP-1,7,1
mmu-miR-181d-5p,miR-181d,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),48,0
hsa-miR-15a,miR-15a,L.donovani,Human,in vitro,PBMC,7,0
hsa-miR-140,miR-140,L.major,Human,in vitro,PBMC,4,1
mmu-miR-495-3p,miR-495,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-miR-23a,miR-23a,L.donovani,Human,in vitro,PBMC,7,0
mmu-miR-466p-3p,miR-466p,L.major,Mouse,in vitro,BMDM (BALB/c females),23,1
mmu-miR-341,miR-341,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
mmu-miR-292-5p,miR-292,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
mmu-miR-5102,miR-5102,L.major,Mouse,in vitro,BMDM (BALB/c females),25,0
mmu-miR-182-5p,miR-182,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),14,1
hsa-let-7b,let-7b,L. donovani,Human,in vitro,THP-1,4,1
hsa-miR-26a-5p,miR-26a,L.major,Human,in vitro,THP-1,4,0
mmu-miR-294-3p,miR-294,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-miR-596,miR-596,L.major,Human,in vitro,PBMC,1,1
mmu-miR-677,miR-677,L.major,Mouse,in vitro,BMDM (BALB/c females),26,0
mmu-miR-3068,miR-3068,L.major,Mouse,in vitro,BMDM (BALB/c females),25,0
mmu-miR-126a-5p,miR-126a,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),47,0
mmu-miR-466q,miR-466q,L.major,Mouse,in vitro,BMDM (BALB/c females),22,1
mmu-miR-1892,miR-1892,L.major,Mouse,in vitro,BMDM (BALB/c females),25,1
hsa-let-7a-3,let-7a,L.major,Human,in vitro,PBMC,3,1
mmu-miR-129-5p,miR-129,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-1945,miR-1945,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-410-3p,miR-410,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),26,1
mmu-miR-466q,miR-466q,L.major,Mouse,in vitro,BMDM (BALB/c females),24,1
mmu-let-7g,let-7g,L.major,Mouse,in vitro,BMDM (BALB/c females),24,1
"mmu-miR-29a,b,c",miR-29a,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),48,1
mmu-miR-3473b,miR-3473b,L.major,Mouse,in vitro,BMDM (BALB/c females),26,0
hsa-let-7f,let-7f,L.donovani,Human,in vitro,PBMC,6,0
hsa-let-7a,let-7a,L. donovani,Human,in vitro,THP-1,3,1
hsa-miR-574-5p,miR-574,L.major,Human,in vitro,PBMC,9,0
mmu-miR-542-5p,miR-542,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-29b,miR-29b,L.major,Human,in vitro,PBMC,7,1
mmu-miR-221,miR-221,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),22,0
hsa-let-7a,let-7a,L.major,Human,in vitro,PBMC,8,0
hsa-miR-30c-5p,miR-30c,L.major,Human,in vitro,THP-1,5,1
hsa-miR-23b,miR-23b,L.major,Human,in vitro,PBMC,9,0
mmu-miR-126a-5p,miR-126a,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),47,0
hsa-miR-30e-5p,miR-30e,L.major,Human,in vitro,THP-1,4,0
hsa-miR-25-3p,miR-25,L.major,Human,in vitro,THP-1,5,1
mmu-miR-294-3p,miR-294,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),22,1
hsa-miR-26a,miR-26a,L.major,Human,in vitro,PBMC,8,0
mmu-miR-677,miR-677,L.major,Mouse,in vitro,BMDM (BALB/c females),26,0
hsa-miR-23a-3p,miR-23a,L.major,Human,in vitro,THP-1,4,1
hsa-miR-24-1,miR-24,L.major,Human,in vitro,PBMC,5,1
hsa-miR-29b,miR-29b,L.donovani,Human,in vitro,PBMC,7,0
hsa-miR-19b,miR-19b,L.major,Human,in vitro,PBMC,5,1
hsa-miR-191,miR-191,L.major,Human,in vitro,PBMC,7,0
hsa-miR-24,miR-24,L.major,Human,in vitro,PBMC,8,1
hsa-miR-338-3p,miR-338,L.donovani,Human,in vitro,PBMC,7,1
mmu-miR-101c,miR-101c,L.major,Mouse,in vitro,BMDM (BALB/c females),24,0
mmu-miR-191,miR-191,L.major,Mouse,in vitro,BMDM (BALB/c females),25,0
mmu-miR-126a-5p,miR-126a,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),48,0
hsa-miR-19a,miR-19a,L.major,Human,in vitro,PBMC,4,0
hsa-miR-21,miR-21,L.donovani,Human,in vitro,PBMC,8,1
mmu-miR-182-5p,miR-182,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),14,1
hsa-miR-519a,miR-519a,L.major,Human,in vitro,PBMC,3,1
hsa-miR-142-3p,miR-142,L.major,Human,in vitro,PBMC,6,1
hsa-miR-210,miR-210,L.major,Human,in vitro,PBMC,8,0
mmu-miR-3075,miR-3075,L.major,Mouse,in vitro,BMDM (BALB/c females),23,1
hsa-miR-146b,miR-146b,L.major,Human,in vitro,PBMC,6,0
mmu-miR-3075,miR-3075,L.major,Mouse,in vitro,BMDM (BALB/c females),24,1
hsa-miR-25,miR-25,L.major,Human,in vitro,PBMC,5,1
mmu-miR-5115,miR-5115,L.major,Mouse,in vitro,BMDM (BALB/c females),26,0
mmu-miR-182-5p,miR-182,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),11,1
hsa-miR-148b,miR-148b,L.donovani,Human,in vitro,PBMC,7,0
mmu-miR-1955-5p,miR-1955,L.major,Mouse,in vitro,BMDM (BALB/c females),23,1
hsa-miR-30a,miR-30a,L.major,Human,in vitro,PBMC,23,0
"mmu-miR-29a,b,c",miR-29a,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),46,1
hsa-miR-16,miR-16,L.donovani,Human,in vitro,PBMC,9,0
hsa-miR-29b,miR-29b,L.donovani,Human,in vitro,PBMC,9,0
hsa-miR-125a-5p,miR-125a,L.donovani,Human,in vitro,PBMC,8,0
hsa-miR-191,miR-191,L.donovani,Human,in vitro,PBMC,10,0
mmu-miR-210,miR-210,L.major,Mouse,in vitro,BMDM (BALB/c females),24,1
hsa-miR-132,miR-132,L.donovani,Human,in vitro,PBMC,9,0
hsa-miR-1260,miR-1260,L.donovani,Human,in vitro,PBMC,6,0
mmu-miR-466p-3p,miR-466p,L.major,Mouse,in vitro,BMDM (BALB/c females),23,1
mmu-let-7g,let-7g,L.major,Mouse,in vitro,BMDM (BALB/c females),26,1
hsa-miR-24-2,miR-24,L.major,Human,in vitro,PBMC,1,1
hsa-miR-221,miR-221,L.major,Human,in vitro,PBMC,23,0
hsa-let-7a,let-7a,L.major,Human,in vitro,PBMC,4,1
hsa-miR-9-5p,miR-9,L.major,Human,in vitro,THP-1,24,1
mmu-miR-294-3p,miR-294,L.amazonensis,Mouse,in vitro,BMDM (BALB/c females),25,1
mmu-miR-6540,miR-6540,L. donovani,Mouse,in vitro,RAW 264.7,29,0
hsa-miR-221-5p,miR-221,L.major,Human,in vitro,THP-1,7,1
hsa-miR-23b,miR-23b,L.major,Human,in vitro,PBMC,7,1
hsa-miR-9-3p,miR-9,L.major,Human,in vitro,THP-1,6,0
mmu-miR-191,miR-191,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-24-1,miR-24,L.major,Human,in vitro,PBMC,1,1
hsa-miR-132-3p,miR-132,L.major,Human,in vitro,THP-1,4,1
mmu-miR-101c,miR-101c,L.major,Mouse,in vitro,BMDM (BALB/c females),23,0
hsa-miR-125a-5p,miR-125a,L.major,Human,in vitro,PBMC,7,0
hsa-miR-191,miR-191,L.donovani,Human,in vitro,PBMC,8,0
mmu-miR-122,miR-122,L. donovani,M ouse,in vitro,Blood serum + liver (BALB/c ),22,0
hsa-miR-26a,miR-26a,L.donovani,Human,in vitro,PBMC,7,0
"""

df = pd.read_csv(io.StringIO(csv_data))
print(f"Données chargées. Taille : {df.shape[0]} lignes.")

# ── Build miRNA lookup BEFORE any processing ──────────────────
# Maps miRNA name → microrna_group_simplified
# Used at prediction time so user only needs to type the miRNA name
mirna_lookup = (
    df[['microrna', 'microrna_group_simplified']]
    .drop_duplicates('microrna')
    .set_index('microrna')['microrna_group_simplified']
    .to_dict()
)
print(f"miRNA lookup built: {len(mirna_lookup)} unique miRNAs")

# ── Scenario ──────────────────────────────────────────────────
df['scenario'] = df['parasite'].str.strip() + "_" + df['cell type'].str.strip()

# ── Features ──────────────────────────────────────────────────
TARGET_COLS = ['microrna_group_simplified', 'parasite', 'organism',
               'cell type', 'scenario']
PASS_COLS   = ['time']
FEATURES    = TARGET_COLS + PASS_COLS

X = df[FEATURES]
y = df['is_upregulated']

# ── Pipeline with OneHotEncoder (honest with OOB) ─────────────
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), TARGET_COLS)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, oob_score=True))
])

print("\nEntraînement du modèle...")
model_pipeline.fit(X, y)
print("Entraînement terminé.")

oob_accuracy = model_pipeline.named_steps['classifier'].oob_score_
print(f"\nScore OOB : {oob_accuracy:.4f} ({oob_accuracy*100:.2f}%)")

# ── Feature importance ────────────────────────────────────────
cat_features_out = (model_pipeline.named_steps['preprocessor']
                    .named_transformers_['cat']
                    .get_feature_names_out(TARGET_COLS))
all_feature_names = list(cat_features_out) + PASS_COLS
importances       = model_pipeline.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature'   : all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 features les plus importantes:")
print(feature_importance_df.head(10).to_string(index=False))

# ── Save model bundle ─────────────────────────────────────────
model_bundle = {
    'model':      model_pipeline,
    'oob_score':  round(oob_accuracy, 4),
    'feature_importance': feature_importance_df.head(10).to_dict('records'),
    'options': {
        'parasite':  sorted(df['parasite'].str.strip().dropna().unique().tolist()),
        'organism':  sorted(df['organism'].dropna().unique().tolist()),
        'cell_type': sorted(df['cell type'].dropna().unique().tolist()),
        'time':      sorted(df['time'].dropna().unique().tolist()),
    },
    'target_cols': TARGET_COLS,
    'pass_cols':   PASS_COLS,
    'mirna_lookup': mirna_lookup,  # miRNA name → microrna_group_simplified
}

model_filename = r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\rf_mirna_model_v1.pkl'
joblib.dump(model_bundle, model_filename)
print(f"\nModèle sauvegardé : '{model_filename}'")
print("\n--- Fin du script ---")

--- Début du script d'entraînement ---
Données chargées. Taille : 149 lignes.
miRNA lookup built: 92 unique miRNAs

Entraînement du modèle...
Entraînement terminé.

Score OOB : 0.8188 (81.88%)

Top 10 features les plus importantes:
                           Feature  Importance
                              time    0.203621
microrna_group_simplified_miR-466p    0.029949
 microrna_group_simplified_miR-142    0.028442
microrna_group_simplified_miR-1955    0.026830
               parasite_L.donovani    0.021019
          scenario_L.donovani_PBMC    0.020869
 microrna_group_simplified_miR-210    0.020405
microrna_group_simplified_miR-3075    0.020111
 microrna_group_simplified_miR-191    0.019515
  microrna_group_simplified_let-7g    0.018305

Modèle sauvegardé : 'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\rf_mirna_model_v1.pkl'

--- Fin du script ---
